# spatialspill tutorial

This notebook runs the full estimator ladder on the bundled subset (Perturb-DBiT sample
mTSG.LV.2, 2,500 pixels, 300 genes) that ships with the repository, then shows how to plug a
squidpy/scanpy AnnData into the same workflow. Everything here runs in under a minute on a
laptop. Numbers printed below are produced by the code cells; the bundled subset is for
demonstration and CI only, not for scientific conclusions (it has 4 NTC pixels).

Prerequisites: `uv sync --extra torch` from a clone of the repository.

In [1]:
import numpy as np
import pandas as pd

from spatialspill.estimators import E2GLM, E1Stratified, NeighbourTTest
from spatialspill.estimators.groups import GroupConfig
from spatialspill.exposure import compute_exposure
from spatialspill.loaders.registry import load_dataset
from spatialspill.outcomes import lognorm
from spatialspill.schema import validate

adata = load_dataset("bundled")
validate(adata)
adata

AnnData object with n_obs × n_vars = 2500 × 300
    obs: 'guide', 'guide_confidence', 'guide_umis', 'guide_umis_total', 'n_guides', 'target', 'is_ntc', 'is_perturbed', 'area', 'cell_type', 'sample', 'batch', 'edge_distance', 'pixel_a', 'pixel_b', 'total_counts'
    uns: 'spatialspill'
    obsm: 'spatial'

## 1. Schema

Every loader produces the same `obs` columns. Coordinates for this dataset are pixel indices
(`uns['spatialspill']['units'] == 'pixel'`), so distance bins below are in pixels.

In [2]:
adata.obs[["guide", "target", "is_ntc", "is_perturbed", "cell_type", "sample"]].head()

,guide,target,is_ntc,is_perturbed,cell_type,sample
mTSG.LV.2:32x30,Kansl1_sg3,Kansl1,False,True,pixel,mTSG.LV.2
mTSG.LV.2:6x50,none,none,False,False,pixel,mTSG.LV.2
mTSG.LV.2:25x22,none,none,False,False,pixel,mTSG.LV.2
mTSG.LV.2:18x12,none,none,False,False,pixel,mTSG.LV.2
mTSG.LV.2:31x34,none,none,False,False,pixel,mTSG.LV.2


## 2. Exposure mapping

Ring counts of every target (and every NTC guide as a pseudo-target) around every unit.

In [3]:
exp = compute_exposure(adata, bins_um=[0, 1.5, 2.5, 3.5])
print(len(exp.targets), "targets incl. NTC pseudo-targets;", exp.n_bins, "rings")
pd.Series(exp.any_perturbed_within).describe()

54 targets incl. NTC pseudo-targets; 3 rings


count    2500.00000
mean        7.80440
std         3.52453
min         0.00000
25%         5.00000
50%         8.00000
75%        10.00000
max        17.00000
dtype: float64

## 3. Estimators

Recipient policy: with 23% of pixels carrying a guide, the clean-control restriction leaves no
controls, so this demo uses all unperturbed pixels (see ADR-003 for why that is not the choice
for single-cell datasets).

In [4]:
Y = lognorm(adata)[:, :50]
names = [str(v) for v in adata.var_names[:50]]
groups = GroupConfig(control_policy="unperturbed", clean_controls=False)
e1 = E1Stratified(n_perm=50, min_cells=5, groups=groups).fit(adata, exp, Y, names).with_fdr()
e2 = E2GLM(min_cells=5, groups=groups, spatial_basis=10).fit(adata, exp, Y, names).with_fdr()
e0 = NeighbourTTest(min_cells=5, groups=groups).fit(adata, exp, Y, names).with_fdr()
for name, df in [("E0 neighbour t-test", e0), ("E1", e1), ("E2 + spatial basis", e2)]:
    i = df[df.identified]
    print(
        f"{name}: {len(i)} identified tests, {(i.qvalue < 0.1).sum()} at q<0.1, "
        f"{(i.pvalue < 0.05).mean():.3f} at p<0.05"
    )

E0 neighbour t-test: 4400 identified tests, 227 at q<0.1, 0.158 at p<0.05
E1: 19600 identified tests, 66 at q<0.1, 0.064 at p<0.05
E2 + spatial basis: 13600 identified tests, 86 at q<0.1, 0.086 at p<0.05


## 4. Calibration check and artifact detector

NTC pseudo-targets should look null. The bleed-through detector projects first-ring spillover
profiles onto autonomous profiles.

In [5]:
from spatialspill.artifacts import artifact_fraction, bleedthrough_table

ntc = e2[e2.identified & e2.target.str.startswith("NTC:")]
print(
    "NTC tests:",
    len(ntc),
    " fraction p<0.05:",
    round((ntc.pvalue < 0.05).mean(), 3) if len(ntc) else "n/a",
)
bt = bleedthrough_table(e2)
artifact_fraction(bt)

NTC tests: 0  fraction p<0.05: n/a


{'n_targets': 34,
 'alpha_ring0': 0.07952638789632357,
 'alpha_last': 0.016037848567869682,
 'r2_ring0': 0.04885702919710293,
 'r2_last': 0.015302167042314465,
 'r2_ring0_minus_last': 0.033554862154788465,
 'frac_targets_alpha_positive_ring0': 0.7941176470588235}

## 5. From a squidpy / scanpy AnnData

Any AnnData with raw counts, `obsm['spatial']` and the obs columns listed in
`spatialspill.schema.OBS_REQUIRED` works. Missing columns can be filled with defaults:

In [6]:
from spatialspill.schema import OBS_REQUIRED, SpatialScreenSchema, edge_distance_from_coords


def adopt(
    ad,
    guide_col,
    target_col,
    ntc_label="NTC",
    sample_col="sample",
    cell_type_col=None,
    technology="custom",
):
    ad = ad.copy()
    ad.obs["guide"] = ad.obs[guide_col].astype(str).fillna("none")
    ad.obs["target"] = ad.obs[target_col].astype(str)
    ad.obs["is_ntc"] = (ad.obs["target"] == ntc_label).to_numpy()
    ad.obs["is_perturbed"] = (~ad.obs["is_ntc"] & (ad.obs["guide"] != "none")).to_numpy()
    ad.obs["guide_confidence"] = np.nan
    ad.obs["area"] = ad.obs["area"] if "area" in ad.obs else np.nan
    ad.obs["cell_type"] = ad.obs[cell_type_col].astype(str) if cell_type_col else "unknown"
    ad.obs["sample"] = ad.obs[sample_col].astype(str)
    ad.obs["batch"] = ad.obs["sample"]
    ad.obs["edge_distance"] = edge_distance_from_coords(
        ad.obsm["spatial"], ad.obs["sample"].to_numpy()
    )
    ad.uns["spatialspill"] = SpatialScreenSchema(technology, "user", "unknown").to_uns()
    validate(ad)
    return ad


print(OBS_REQUIRED)

('guide', 'guide_confidence', 'target', 'is_ntc', 'is_perturbed', 'area', 'cell_type', 'sample', 'batch', 'edge_distance')
